# 🎬 Cắt frame từ video (.mp4) — Google Colab

- ⏱️ **Tùy chỉnh số frame/giây** muốn cắt (slider)
- ➕ **Nút thêm nhiều video** cùng lúc
- 📦 Tự động **zip + tải về** sau khi cắt

Có 2 cách:
- **Cách 1 (UI bấm nút)** — tiện cho video nhỏ/vừa (< ~200MB/file)
- **Cách 2 (Google Drive)** — cho video LỚN, không giới hạn dung lượng

`Runtime → Run all` rồi dùng giao diện ở Cách 1.

In [ ]:
# 1 — Setup + hàm cắt frame (có callback tiến trình)
import cv2, os, shutil, zipfile
from pathlib import Path

VIDEO_DIR = '/content/_videos'
FRAME_DIR = '/content/_frames'
os.makedirs(VIDEO_DIR, exist_ok=True)
os.makedirs(FRAME_DIR, exist_ok=True)

def extract_frames(video_path, out_dir, target_fps, resize=None, jpg_quality=92, on_progress=None):
    """Cắt frame: lấy ~target_fps khung/giây. resize=(w,h) nếu muốn resize.
    on_progress(cur, total): callback cập nhật tiến trình."""
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0, 0, 0
    native = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step   = max(1, round(native / max(target_fps, 0.01)))   # lấy mỗi 'step' khung
    stem   = Path(video_path).stem
    saved, idx = 0, 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if idx % step == 0:
            if resize: frame = cv2.resize(frame, resize)
            cv2.imwrite(f'{out_dir}/{stem}_{saved:05d}.jpg', frame,
                        [cv2.IMWRITE_JPEG_QUALITY, jpg_quality])
            saved += 1
        idx += 1
        if on_progress and idx % 15 == 0:
            on_progress(idx, total)
    cap.release()
    if on_progress: on_progress(total, total)
    return saved, native, total

print('✅ Sẵn sàng. Chạy cell tiếp theo để hiện giao diện.')

---
## 🖱️ Cách 1 — Giao diện bấm nút
1. Kéo **Frame/giây** muốn cắt  
2. Bấm **➕ Thêm video** → chọn nhiều file .mp4  
3. Bấm **✂️ Cắt frame** → tự cắt + zip + tải về

In [ ]:
# 2 — Giao diện (có THANH TIẾN TRÌNH + trạng thái + báo lỗi rõ)
import ipywidgets as widgets
from IPython.display import display
from google.colab import files
import traceback

fps_w    = widgets.BoundedFloatText(value=2.0, min=0.1, max=60, step=0.5,
                                    description='Frame/giây:', style={'description_width':'initial'})
resize_w = widgets.Dropdown(options=[('Giữ nguyên', None), ('64×64 (CNN)', (64,64)),
                                     ('224×224', (224,224)), ('640×640 (YOLO)', (640,640))],
                            value=None, description='Resize:')
upload_w = widgets.FileUpload(accept='.mp4,.avi,.mov,.mkv', multiple=True,
                             description='➕ Thêm video')
run_btn  = widgets.Button(description='✂️ Cắt frame', button_style='success', icon='scissors')

# Hiển thị tiến trình bằng widget value (chắc chắn render trên Colab, KHÔNG dùng Output)
progress = widgets.IntProgress(value=0, min=0, max=100, description='Tiến trình:',
                               bar_style='info', layout=widgets.Layout(width='90%'))
status   = widgets.HTML(value='<i>Chưa chạy. Thêm video rồi bấm ✂️ Cắt frame.</i>')
log      = widgets.HTML(value='')

# Chuẩn hoá value FileUpload (ipywidgets 7 vs 8)
def _norm(items):
    if isinstance(items, dict):
        return [(n, d['content']) for n, d in items.items()]
    return [(d['name'], d['content']) for d in items]

def on_run(_):
    try:
        flist = _norm(upload_w.value)
        if not flist:
            status.value = '⚠️ <b>Chưa thêm video.</b> Bấm ➕ Thêm video → chọn file → RỒI bấm ✂️ Cắt frame.'
            return
        shutil.rmtree(FRAME_DIR, ignore_errors=True); os.makedirs(FRAME_DIR, exist_ok=True)
        lines, total_saved, n = [], 0, len(flist)
        for i, (name, content) in enumerate(flist):
            status.value = f'🎬 Đang xử lý <b>{i+1}/{n}</b>: {name}'
            progress.bar_style = 'info'; progress.value = 0
            vp = f'{VIDEO_DIR}/{name}'
            with open(vp, 'wb') as f: f.write(content)
            sub = f'{FRAME_DIR}/{Path(name).stem}'
            def cb(cur, tot):
                progress.max = max(tot, 1); progress.value = min(cur, tot)
            saved, native, total = extract_frames(vp, sub, fps_w.value, resize_w.value, on_progress=cb)
            total_saved += saved
            lines.append(f'✅ {name}: cắt <b>{saved}</b> frame (gốc {native:.0f} fps, {total} khung)')
            log.value = '<br>'.join(lines)
        if total_saved == 0:
            status.value = '❌ Không cắt được frame — file video có hợp lệ không?'
            progress.bar_style = 'danger'; return
        status.value = f'📦 Đang nén {total_saved} frame...'
        shutil.make_archive('/content/frames_output', 'zip', FRAME_DIR)
        progress.bar_style = 'success'
        status.value = f'✅ <b>XONG! Tổng {total_saved} frame.</b> Đang tải frames_output.zip về máy...'
        files.download('/content/frames_output.zip')
    except Exception as e:
        status.value = f'❌ <b>Lỗi:</b> {e}'
        log.value = '<pre style="color:red">' + traceback.format_exc() + '</pre>'

run_btn.on_click(on_run)
display(widgets.VBox([widgets.HBox([fps_w, resize_w]), upload_w, run_btn, progress, status, log]))
print('👇 Giao diện ở dưới. Khi bấm Cắt frame, xem THANH TIẾN TRÌNH + dòng trạng thái.')

---
## 💾 Cách 2 — Google Drive (cho video LỚN)
Bỏ video vào thư mục Drive rồi chạy cell dưới (không giới hạn dung lượng).

In [ ]:
# 3 — Cắt frame từ video trong Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ↓ CHỈNH 2 dòng này
DRIVE_VIDEO_FOLDER = '/content/drive/MyDrive/videos'      # thư mục chứa video
TARGET_FPS         = 2.0                                   # frame/giây muốn cắt
RESIZE             = None                                  # None | (64,64) | (640,640)

OUT = '/content/drive/MyDrive/extracted_frames'
os.makedirs(OUT, exist_ok=True)
vids = [p for p in Path(DRIVE_VIDEO_FOLDER).glob('*') if p.suffix.lower() in ('.mp4','.avi','.mov','.mkv')]
print(f'Tìm thấy {len(vids)} video trong {DRIVE_VIDEO_FOLDER}\n')

grand = 0
for v in vids:
    sub = f'{OUT}/{v.stem}'
    saved, native, total = extract_frames(str(v), sub, TARGET_FPS, RESIZE)
    grand += saved
    print(f'✅ {v.name}: {saved} frame  (gốc {native:.0f} fps)')
print(f'\n📁 Tổng {grand} frame → lưu tại {OUT} (trên Drive)')

---
## 💡 Mẹo
- **Frame/giây bao nhiêu?** Buồn ngủ là trạng thái chậm → **2–5 fps là đủ**, tránh trùng ảnh. Muốn nhiều dữ liệu hơn thì tăng lên.
- **Resize 64×64** nếu cắt để train CNN mắt/ngáp; **640×640** nếu để train YOLO.
- Sau khi cắt: dùng `colab_extract_frames` → tải zip → giải nén → xếp vào folder theo class (`eyes_open/`, `eyes_closed/`...) → đưa vào pipeline (đặt `already_cropped=False` để tự crop mặt → mắt/miệng).
- Video lớn nên dùng **Cách 2 (Drive)** vì nút upload giới hạn ~200MB/file.